# HAM10000 — Exploratory Data Analysis

This notebook walks through:
1. Loading and inspecting the HAM10000 metadata
2. Class distribution analysis
3. Image sample visualisation
4. Pixel statistics (mean / std per channel)
5. Train/val/test split verification
6. Quick sanity-check inference with a random-weight model

In [ ]:
import sys, os
from pathlib import Path

# Make the backend package importable from the notebook
BACKEND_DIR = Path(os.getcwd()).parent.parent
sys.path.insert(0, str(BACKEND_DIR))
print('Backend root:', BACKEND_DIR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load Metadata

In [ ]:
META_CSV = BACKEND_DIR / 'data' / 'raw' / 'HAM10000_metadata.csv'

if not META_CSV.exists():
    print(f'Metadata not found at {META_CSV}')
    print('Run: python ml/download_dataset.py')
else:
    df = pd.read_csv(META_CSV)
    print(f'Rows: {len(df):,}  |  Columns: {list(df.columns)}')
    df.head()

In [ ]:
# Basic stats
df.info()
df.describe(include='all')

## 2. Class Distribution

In [ ]:
LABEL_NAMES = {
    'nv':    'Melanocytic nevi',
    'mel':   'Melanoma',
    'bkl':   'Benign keratosis',
    'bcc':   'Basal cell carcinoma',
    'akiec': 'Actinic keratoses',
    'vasc':  'Vascular lesions',
    'df':    'Dermatofibroma',
}

counts = df['dx'].map(LABEL_NAMES).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
counts.plot(kind='barh', ax=axes[0], color=sns.color_palette('muted', len(counts)))
axes[0].set_title('Sample Count per Class', fontsize=13)
axes[0].set_xlabel('Count')
for i, v in enumerate(counts):
    axes[0].text(v + 20, i, str(v), va='center')

# Pie chart
axes[1].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('muted', len(counts)), startangle=140)
axes[1].set_title('Class Proportion', fontsize=13)

plt.suptitle('HAM10000 Class Distribution', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print(f'\nClass imbalance ratio (max/min): {counts.max()/counts.min():.1f}x')

## 3. Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Age distribution
df['age'].dropna().hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')

# Sex distribution
df['sex'].value_counts().plot(kind='bar', ax=axes[1], color=['#4C72B0','#DD8452'], edgecolor='white')
axes[1].set_title('Sex Distribution')
axes[1].set_xlabel('')

# Localization
df['localization'].value_counts().head(10).plot(
    kind='barh', ax=axes[2], color=sns.color_palette('pastel', 10))
axes[2].set_title('Top-10 Lesion Locations')

plt.tight_layout()
plt.show()

## 4. Sample Images per Class

In [ ]:
from ml.data_loader import _find_image

SAMPLES_PER_CLASS = 4
classes = list(LABEL_NAMES.keys())

fig, axes = plt.subplots(len(classes), SAMPLES_PER_CLASS,
                          figsize=(SAMPLES_PER_CLASS * 3, len(classes) * 3))

for row_idx, cls in enumerate(classes):
    samples = df[df['dx'] == cls].sample(min(SAMPLES_PER_CLASS, len(df[df['dx'] == cls])),
                                         random_state=42)
    for col_idx, (_, row) in enumerate(samples.iterrows()):
        ax = axes[row_idx][col_idx]
        img_path = _find_image(row['image_id'])
        if img_path:
            ax.imshow(Image.open(img_path))
        else:
            ax.imshow(np.zeros((100, 100, 3), dtype=np.uint8))
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(LABEL_NAMES[cls], fontsize=9, rotation=0,
                          labelpad=80, va='center')

plt.suptitle('Sample Images per Class', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Pixel Statistics (Mean & Std per Channel)

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader
from ml.data_loader import HAM10000Dataset
import torch

# Use a small subset for speed
subset_df = df.sample(500, random_state=42)

simple_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

ds = HAM10000Dataset(subset_df, transform=simple_transform)
loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)

mean = torch.zeros(3)
std  = torch.zeros(3)
n    = 0

for imgs, _ in loader:
    b = imgs.size(0)
    imgs = imgs.view(b, 3, -1)
    mean += imgs.mean(dim=2).sum(dim=0)
    std  += imgs.std(dim=2).sum(dim=0)
    n    += b

mean /= n
std  /= n

print(f'Channel mean: R={mean[0]:.4f}  G={mean[1]:.4f}  B={mean[2]:.4f}')
print(f'Channel std : R={std[0]:.4f}  G={std[1]:.4f}  B={std[2]:.4f}')

## 6. Train / Val / Test Split Verification

In [ ]:
from ml.data_loader import get_dataloaders

train_loader, val_loader, test_loader, class_weights = get_dataloaders(batch_size=32, num_workers=0)

print(f'Train batches : {len(train_loader):>5}  ({len(train_loader.dataset):>6} samples)')
print(f'Val   batches : {len(val_loader):>5}  ({len(val_loader.dataset):>6} samples)')
print(f'Test  batches : {len(test_loader):>5}  ({len(test_loader.dataset):>6} samples)')
print(f'\nClass weights : {class_weights.tolist()}')

## 7. Sanity-Check Inference (Random Weights)

In [ ]:
from app.core.classifier import Classifier
from app.config import settings
import io

clf = Classifier()

# Grab first image from test set
imgs, labels = next(iter(test_loader))

# Convert first tensor back to PIL and save to a temp buffer
from app.core.preprocessing import tensor_to_numpy

arr = tensor_to_numpy(imgs[0])
pil_img = Image.fromarray(arr)
buf = io.BytesIO()
pil_img.save(buf, format='JPEG')
buf.seek(0)

pred_idx, confidence, all_probs = clf.predict(buf.getvalue())

print(f'True label  : {settings.CLASS_NAMES[labels[0].item()]}')
print(f'Predicted   : {settings.CLASS_NAMES[pred_idx]}  ({confidence*100:.1f}%)')
print('\nAll probabilities:')
for name, prob in zip(settings.CLASS_NAMES, all_probs):
    bar = '█' * int(prob * 30)
    print(f'  {name:<35} {prob:.4f}  {bar}')

## 8. Grad-CAM Preview

In [ ]:
from app.core.gradcam import GradCAM
import tempfile, os

gradcam = GradCAM()

# Save the test image to a temp file (GradCAM needs a file path)
with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as f:
    pil_img.save(f.name)
    tmp_path = f.name

try:
    overlay = gradcam.overlay(tmp_path, class_idx=pred_idx)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(arr);          axes[0].set_title('Original');        axes[0].axis('off')
    cam = gradcam.generate(tmp_path, class_idx=pred_idx)
    axes[1].imshow(cam, cmap='jet'); axes[1].set_title('Grad-CAM (raw)'); axes[1].axis('off')
    axes[2].imshow(overlay);     axes[2].set_title('Overlay');          axes[2].axis('off')

    plt.suptitle(f'Grad-CAM — Predicted: {settings.CLASS_NAMES[pred_idx]}', fontsize=13)
    plt.tight_layout()
    plt.show()
finally:
    os.unlink(tmp_path)